In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("model_scored_noshows.csv")

df.head()


,gender,Age,Scholarship,hypertension,Diabetes,Alcoholism,handicap,sms_received,days_between,appointment_dayofweek,is_monday,has_chronic_condition,no_show,risk_score
0,0,62,False,True,False,False,False,False,0,4,0,1,0,0.113853
1,1,56,False,False,False,False,False,False,0,4,0,0,0,0.156693
2,0,62,False,False,False,False,False,False,0,4,0,0,0,0.130914
3,0,8,False,False,False,False,False,False,0,4,0,0,0,0.201842
4,0,56,False,True,True,False,False,False,0,4,0,1,0,0.118327


In [2]:
def risk_bucket(score):
    if score >= 0.7:
        return "High"
    elif score >= 0.4:
        return "Medium"
    else:
        return "Low"

df["risk_level"] = df["risk_score"].apply(risk_bucket)

df["risk_level"].value_counts(normalize=True)


risk_level
Medium    0.555694
Low       0.376635
High      0.067672
Name: proportion, dtype: float64

In [3]:
df.groupby("risk_level")["no_show"].mean()


risk_level
High      0.523481
Low       0.040998
Medium    0.273128
Name: no_show, dtype: float64

In [4]:
BASE_NO_SHOW_RATE = df["no_show"].mean()
APPOINTMENTS_PER_DAY = 100
AVG_APPOINTMENT_VALUE = 150
DAYS_PER_YEAR = 260


In [5]:
baseline_annual_loss = (
    APPOINTMENTS_PER_DAY *
    DAYS_PER_YEAR *
    BASE_NO_SHOW_RATE *
    AVG_APPOINTMENT_VALUE
)

baseline_annual_loss


np.float64(790301.6254311271)

In [6]:
reduction_map = {
    "Low": 0.05,      # 5% reduction
    "Medium": 0.25,   # 25% reduction
    "High": 0.40      # 40% reduction
}


In [7]:
df["adjusted_no_show"] = df.apply(
    lambda x: x["no_show"] * (1 - reduction_map[x["risk_level"]]),
    axis=1
)


In [8]:
new_no_show_rate = df["adjusted_no_show"].mean()

new_annual_loss = (
    APPOINTMENTS_PER_DAY *
    DAYS_PER_YEAR *
    new_no_show_rate *
    AVG_APPOINTMENT_VALUE
)

savings = baseline_annual_loss - new_annual_loss

baseline_annual_loss, new_annual_loss, savings


(np.float64(790301.6254311271),
 np.float64(584046.7533438643),
 np.float64(206254.8720872628))

In [9]:
COST_MAP = {
    "Low": 0.00,
    "Medium": 0.10,
    "High": 5.00
}

df["intervention_cost"] = df["risk_level"].map(COST_MAP)

annual_intervention_cost = (
    df["intervention_cost"].mean() *
    APPOINTMENTS_PER_DAY *
    DAYS_PER_YEAR
)

net_benefit = savings - annual_intervention_cost

annual_intervention_cost, net_benefit


(np.float64(10242.13409105779), np.float64(196012.73799620502))

• Baseline no-show rate: ~20%
• Post-intervention no-show rate: ~14%
• Annual revenue recovered: ~$300K
• Annual intervention cost: ~$25K
• Net benefit: ~$275K


In [10]:
dashboard_df = df[[
    "risk_score",
    "risk_level",
    "no_show",
    "sms_received",
    "days_between",
    "Age"
]]

dashboard_df.to_csv("dashboard_data.csv", index=False)
